In [1]:
import csv
f = open("fixed_cost.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [2]:
print(headers)

['Machine', 'Fixed Cost', 'Variable Cost', 'Capacity']


In [3]:
FixedCost = {}
VariableCost = {}
Capacity = {}
for row in table:
    FixedCost[row['Machine']] = float(row['Fixed Cost'])
    VariableCost[row['Machine']] = float(row['Variable Cost'])
    Capacity[row['Machine']] = float(row['Capacity'])
    
# demand
Demand = 1800

In [4]:
FixedCost

{'M1': 1000.0, 'M2': 950.0, 'M3': 800.0, 'M4': 850.0, 'M5': 900.0, 'M6': 700.0}

In [5]:
VariableCost

{'M1': 21.0, 'M2': 23.0, 'M3': 25.0, 'M4': 24.0, 'M5': 20.0, 'M6': 26.0}

In [6]:
Capacity

{'M1': 500.0, 'M2': 600.0, 'M3': 750.0, 'M4': 400.0, 'M5': 600.0, 'M6': 800.0}

In [7]:
from docplex.mp.model import Model
from docplex.util.environment import get_environment
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [8]:
from docplex.mp.model import Model
mdl = Model()

In [9]:
# variables
select = mdl.binary_var_dict(FixedCost, name='Machine')
produce = mdl.continuous_var_dict(VariableCost, name='Produce')

In [10]:
# objective
TotalFixedCost = mdl.sum(FixedCost[i]*select[i] for i in select)
TotalVariableCost = mdl.sum(VariableCost[i]*produce[i] for i in produce)
TotalCost = TotalFixedCost + TotalVariableCost

mdl.minimize(TotalCost)

mdl.add_kpi(TotalFixedCost, 'total fixed cost')
mdl.add_kpi(TotalVariableCost, 'total variable cost')
mdl.add_kpi(TotalCost, 'total cost')

DecisionKPI(name=total cost,expr=1000Machine_M1+950Machine_M2+800Machine_M3+850Machine_M4+900Mach..)

In [11]:
# constraints: capacity/linkage
for i in select:
    mdl.add_constraint(produce[i] <= Capacity[i]*select[i])

In [12]:
# constraint: meet demand
mdl.add_constraint( mdl.sum(produce[i] for i in produce) >= Demand )

docplex.mp.LinearConstraint[](Produce_M1+Produce_M2+Produce_M3+Produce_M4+Produce_M5+Produce_M6,GE,1800)

In [13]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.048167,status='integer optimal solution')

In [14]:
mdl.print_solution()

objective: 42400.000
status: OPTIMAL_SOLUTION(2)
  Machine_M1=1
  Machine_M2=1
  Machine_M4=1
  Machine_M5=1
  Produce_M1=500.000
  Produce_M2=600.000
  Produce_M4=100.000
  Produce_M5=600.000


In [15]:
mdl.report()

* model docplex_model2 solved with objective = 42400.000
*  KPI: total fixed cost    = 3700.000
*  KPI: total variable cost = 38700.000
*  KPI: total cost          = 42400.000


In [15]:
mdl.report()

* model docplex_model2 solved with objective = 42400.000
*  KPI: total fixed cost    = 3700.000
*  KPI: total variable cost = 38700.000
*  KPI: total cost          = 42400.000
